# fashionAI — explore embeddings & HNSW

Run `python data/prepare_dataset.py --max-samples 500` first.

In [2]:
from pathlib import Path
import json
import sys
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt

ROOT = Path('..').resolve()
sys.path.insert(0, str(ROOT))
DATA = ROOT / 'data'

emb = np.load(DATA / 'embeddings.npy')
meta = json.loads((DATA / 'metadata.json').read_text())
print(emb.shape, len(meta))

FileNotFoundError: [Errno 2] No such file or directory: '/Users/moosairfaan/fashionAI/artifacts/embeddings.npy'

In [ ]:
from index.brute_force import BruteForceIndex
from index.hnsw import HNSWIndex

brute = BruteForceIndex(emb, meta)
hnsw_path = DATA / 'hnsw.pkl'
hnsw = HNSWIndex.load(hnsw_path) if hnsw_path.exists() else None
if hnsw is None:
    hnsw = HNSWIndex(dim=emb.shape[1], M=16, ef_construction=80, num_layers=4)
    for i, v in enumerate(emb):
        hnsw.insert(v, id=int(meta[i]['id']))

qid = 0
k = 6
truth = brute.search(emb[qid], k=k)
approx = hnsw.search(emb[qid], k=k, ef_search=50)
print('brute', [h['id'] for h in truth])
print('hnsw ', [h['id'] for h in approx])
overlap = set(h['id'] for h in truth) & set(h['id'] for h in approx)
print(f'recall@{k} =', len(overlap) / k)

In [ ]:
by_id = {int(row['id']): row for row in meta}
fig, axes = plt.subplots(1, k + 1, figsize=(2.2 * (k + 1), 3))
axes[0].imshow(Image.open(DATA / meta[qid]['path']))
axes[0].set_title('query')
axes[0].axis('off')
for ax, hit in zip(axes[1:], approx):
    ax.imshow(Image.open(DATA / by_id[hit['id']]['path']))
    ax.set_title(f"{hit['score']:.2f}")
    ax.axis('off')
plt.tight_layout()
plt.show()